# Exploración de datos (EDA): churn de clientes de telecomunicaciones

Fase *Data Understanding* de CRISP-DM. El objetivo es conocer los datos antes de
limpiarlos y modelar: estructura, calidad (faltantes, duplicados, outliers),
la variable objetivo y qué variables se relacionan con el churn.

Las decisiones que salgan de aquí se implementarán en los pipelines de Kedro.

**Pregunta de negocio:** ¿qué clientes tienen mayor probabilidad de cancelar su
servicio, para que el equipo de retención los contacte primero?
(ver `docs/problema_de_negocio.md`)

## 1. Configuración y carga de datos

Los datos se cargan desde el **Data Catalog de Kedro** (`conf/base/catalog.yml`),
no con rutas escritas a mano. Así el notebook usa exactamente las mismas fuentes
que los pipelines.

Para ejecutarlo: abrir el notebook con el entorno del proyecto activo
(`jupyter lab` o VS Code) desde la carpeta `notebooks/` o la raíz del proyecto.

In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from kedro.config import OmegaConfigLoader
from kedro.io import DataCatalog

# Aviso interno de Kedro/OmegaConf que no afecta el análisis
warnings.filterwarnings("ignore", message=".*register_new_resolver.*")

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

In [2]:
# Trabajar desde la raíz del proyecto para que las rutas del catálogo funcionen
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(raiz)

# Leer el Data Catalog de Kedro (conf/base/catalog.yml)
config = OmegaConfigLoader(conf_source="conf", base_env="base", default_run_env="local")
catalog = DataCatalog.from_config(config["catalog"])
catalog.keys()

['raw_demographics',
 'raw_location',
 'raw_population',
 'raw_services',
 'raw_status',
 'raw_telco_churn',
 'raw_customer_churn']

In [3]:
demographics = catalog.load("raw_demographics")
location = catalog.load("raw_location")
population = catalog.load("raw_population")
services = catalog.load("raw_services")
status = catalog.load("raw_status")

tablas = {
    "demographics": demographics,
    "location": location,
    "population": population,
    "services": services,
    "status": status,
}
pd.DataFrame(
    {
        nombre: {"filas": df.shape[0], "columnas": df.shape[1]}
        for nombre, df in tablas.items()
    }
).T

,filas,columnas
demographics,7043,9
location,7043,10
population,1671,3
services,7043,31
status,7043,12


## 2. Estructura y tipos de cada tabla

Para cada tabla revisamos el tipo de dato de cada columna, cuántos valores
distintos tiene y un ejemplo. Con esto detectamos:

- columnas **identificadoras** (un valor distinto por fila), que no sirven para predecir;
- columnas **constantes** (un solo valor), que no aportan información;
- columnas con un **tipo incorrecto** (por ejemplo, números guardados como texto).

In [4]:
def resumen_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, número de valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame(
        {
            "tipo": df.dtypes.astype(str),
            "valores_distintos": df.nunique(),
            "ejemplo": df.iloc[0],
        }
    )

In [5]:
from IPython.display import Markdown

for nombre, df in tablas.items():
    display(Markdown(f"**{nombre}**: {df.shape[0]} filas x {df.shape[1]} columnas"))
    display(resumen_columnas(df))

**demographics**: 7043 filas x 9 columnas

,tipo,valores_distintos,ejemplo
Customer ID,str,7043,8779-QRDMV
Count,int64,1,1
Gender,str,2,Male
Age,int64,62,78
Under 30,str,2,No
Senior Citizen,str,2,Yes
Married,str,2,No
Dependents,str,2,No
Number of Dependents,int64,10,0


**location**: 7043 filas x 10 columnas

,tipo,valores_distintos,ejemplo
Location ID,str,7043,OXCZEW7397
Customer ID,str,7043,8779-QRDMV
Count,int64,1,1
Country,str,1,United States
State,str,1,California
City,str,1106,Los Angeles
Zip Code,int64,1625,90022
Lat Long,str,1679,"34.02381, -118.156582"
Latitude,float64,1626,34.02381
Longitude,float64,1625,-118.156582


**population**: 1671 filas x 3 columnas

,tipo,valores_distintos,ejemplo
ID,int64,1671,1
Zip Code,int64,1671,90001
Population,str,1607,"54,492"


**services**: 7043 filas x 31 columnas

,tipo,valores_distintos,ejemplo
Service ID,str,7043,IJKDQVSWH3522
Customer ID,str,7043,8779-QRDMV
Count,int64,1,1
Quarter,str,1,Q3
Referred a Friend,str,2,No
Number of Referrals,int64,12,0
Tenure in Months,int64,72,1
Offer,str,5,NaN
Phone Service,str,2,No
Avg Monthly Long Distance Charges,float64,3584,0.0


**status**: 7043 filas x 12 columnas

,tipo,valores_distintos,ejemplo
Status ID,str,7043,SWSORB1252
Customer ID,str,7043,8779-QRDMV
Count,int64,1,1
Quarter,str,1,Q3
Satisfaction Score,int64,5,3
Customer Status,str,3,Churned
Churn Label,str,2,Yes
Churn Value,int64,2,1
Churn Score,int64,81,91
CLTV,int64,3438,5433


### Columnas constantes e identificadores

Buscamos de forma automática las columnas con un solo valor y las que tienen un
valor distinto en cada fila (posibles IDs).

In [6]:
filas = []
for nombre, df in tablas.items():
    for col in df.columns:
        distintos = df[col].nunique()
        if distintos == 1:
            filas.append((nombre, col, "constante", df[col].iloc[0]))
        elif distintos == len(df) and df[col].dtype != float:
            filas.append((nombre, col, "identificador", df[col].iloc[0]))

pd.DataFrame(filas, columns=["tabla", "columna", "tipo", "ejemplo"])

,tabla,columna,tipo,ejemplo
0,demographics,Customer ID,identificador,8779-QRDMV
1,demographics,Count,constante,1
2,location,Location ID,identificador,OXCZEW7397
3,location,Customer ID,identificador,8779-QRDMV
4,location,Count,constante,1
5,location,Country,constante,United States
6,location,State,constante,California
7,population,ID,identificador,1
8,population,Zip Code,identificador,90001
9,services,Service ID,identificador,IJKDQVSWH3522


### Población guardada como texto

`Population` debería ser numérica, pero se cargó como texto porque usa comas
como separador de miles, y en algunos casos con el formato de agrupación de la
India en 3 códigos postales (por ejemplo `1,01,215` = 101,215).

In [7]:
population["Population"].head(10).tolist()

['54,492',
 '44,586',
 '58,198',
 '67,852',
 '43,019',
 '62,784',
 '45,025',
 '30,852',
 '1,957',
 '1,01,215']

In [8]:
# Valores con el formato de agrupación de la India (más de una coma antes de los últimos 3 dígitos)
formato_india = population["Population"].str.match(r"^\d{1,2},\d{2},\d{3}$")
population.loc[formato_india, "Population"].head()

9      1,01,215
62     1,05,285
125    1,03,214
Name: Population, dtype: str

In [9]:
# Al quitar todas las comas el número queda correcto en ambos formatos
pd.to_numeric(population["Population"].str.replace(",", "")).describe()

count      1671.000000
mean      20276.384201
std       20689.117300
min          11.000000
25%        1789.000000
50%       14239.000000
75%       32942.500000
max      105285.000000
Name: Population, dtype: float64

### Observaciones

1. **Todas las tablas de clientes tienen 7,043 filas** y un `Customer ID` único,
   así que se pueden unir uno a uno (se valida en la sección 3).
2. **Columnas sin información para el modelo:**
   - Constantes: `Count`, `Country` (*United States*), `State` (*California*) y
     `Quarter` (*Q3*). Todos los datos son de un mismo trimestre y estado.
   - Identificadores internos de cada tabla: `Location ID`, `Service ID`,
     `Status ID` e `ID`. Solo `Customer ID` se conserva, como llave.
3. **Tipos a corregir:** `Population` es texto y se convierte a número quitando
   las comas (rango: 11 a 105,285 habitantes por código postal).
4. **Columnas redundantes** (la misma información en dos columnas):
   - `Under 30` y `Senior Citizen` se derivan de `Age`.
   - `Dependents` se deriva de `Number of Dependents`.
   - `Lat Long` repite `Latitude` y `Longitude`.
   - `Churn Label` repite `Churn Value`.
5. **Muchas variables categóricas Sí/No** (servicios contratados), que se
   codificarán como 0/1. `Contract`, `Payment Method`, `Internet Type` y
   `Offer` tienen 3 a 5 categorías.
6. **`Offer` tiene valores vacíos.** Probablemente significan *el cliente no
   aceptó ninguna oferta*; se confirma en la sección de faltantes.
7. A diferencia de otras versiones de este dataset, `Total Charges` ya viene
   como número.